# Numerical Methods for Bioinformatics

## Learning Objectives

By the end of this module you will be able to:

- Implement Lagrange and Newton interpolation polynomials and explain the Runge phenomenon
- Apply cubic spline interpolation to reconstruct missing time points in biological time series
- Compute numerical derivatives (forward, backward, central differences) and understand their error properties
- Use the trapezoidal rule and Simpson's rule to approximate definite integrals; compute area under biological curves
- Solve linear and nonlinear curve-fitting problems with numpy.linalg.lstsq and scipy.optimize.curve_fit
- Fit Michaelis-Menten, Hill, and four-parameter logistic (4PL) models and assess goodness-of-fit with R², AIC, and chi-squared
- Implement gradient descent from scratch and use scipy.optimize.minimize for maximum likelihood estimation
- Apply the Fast Fourier Transform to detect periodic signals in gene expression and spectral data

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


[← Previous: 3.15 Population Genetics](../15_Population_Genetics_and_Molecular_Evolution/01_population_genetics_and_molecular_evolution.ipynb) | [Next: 3.17 Genome Assembly →](../17_Genome_Assembly_and_Advanced_NGS/01_genome_assembly_and_advanced_ngs.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, interp1d
from scipy.integrate import quad, trapezoid, simpson
from scipy.optimize import curve_fit, minimize
from scipy.stats import chi2

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

---
## Part 1: Interpolation

Interpolation builds a function that passes exactly through a set of data points (x_1, y_1), ..., (x_n, y_n). This is different from curve fitting, where the function is allowed to deviate from the data to avoid overfitting noise.

**Key theoretical result (Weierstrass, 1885):** Any continuous function on [a, b] can be approximated to arbitrary precision by a polynomial. However, *which* polynomial matters enormously.

### 1.1 Lagrange Interpolation

Given n distinct nodes x_1, ..., x_n, the **Lagrange interpolation polynomial** of degree n-1 is:

    pi_n(x) = sum_i y_i * l_i(x),   l_i(x) = prod_{j != i} (x - x_j) / (x_i - x_j)

Each basis polynomial l_i equals 1 at x_i and 0 at all other nodes, so pi_n(x_i) = y_i is guaranteed.

**Error bound:** For f in W^n[M_n, I] (i.e., |f^(n)| <= M_n on I = [a,b]):

    |f(x) - pi_n(x)| = f^(n)(xi) / n! * prod_i (x - x_i),   xi in [a, b]

In [ ]:
def lagrange_interpolate(x_nodes, y_nodes, x_eval):
    """Evaluate the Lagrange interpolation polynomial at x_eval."""
    x_nodes = np.asarray(x_nodes, float)
    y_nodes = np.asarray(y_nodes, float)
    x_eval  = np.asarray(x_eval, float)
    n = len(x_nodes)
    result = np.zeros_like(x_eval)
    for i in range(n):
        basis = np.ones_like(x_eval)
        for j in range(n):
            if j != i:
                basis *= (x_eval - x_nodes[j]) / (x_nodes[i] - x_nodes[j])
        result += y_nodes[i] * basis
    return result


# Mendeleev 1881 solubility data: NaNO2 dissolved in 100 mL water vs temperature
T_data = np.array([0, 4, 10, 15, 21, 29, 36, 51, 68], dtype=float)
S_data = np.array([66.7, 71.0, 76.3, 80.6, 85.7, 99.4, 99.4, 113.6, 125.1])

T_fine = np.linspace(0, 68, 300)
S_lag  = lagrange_interpolate(T_data, S_data, T_fine)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(T_data, S_data, zorder=5, color='black', label='Mendeleev data')
ax.plot(T_fine, S_lag, label='Lagrange polynomial (degree 8)')
ax.set_xlabel('Temperature (C)')
ax.set_ylabel('NaNO2 per 100 mL water (g)')
ax.set_title('Lagrange interpolation of solubility data')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Interpolated solubility at 40 C: {lagrange_interpolate(T_data, S_data, [40])[0]:.1f} g')

### 1.2 The Runge Phenomenon and Chebyshev Nodes

High-degree Lagrange interpolation on a **uniform grid** suffers from the **Runge phenomenon**: oscillations near the interval endpoints grow exponentially with the polynomial degree. Runge (1901) demonstrated this on f(x) = 1/(1+25x^2) with a degree-20 polynomial.

The fix is to use **Chebyshev nodes** which cluster near the endpoints:

    x_k = (b+a)/2 + (b-a)/2 * cos(pi*(2k-1)/(2n)),   k = 1, ..., n

These are zeros of the Chebyshev polynomial T_n(y) = cos(n * arccos(y)), and they minimise max_x |omega_n(x, X)| over all n-point grids.

In [ ]:
def chebyshev_nodes(n, a=-1.0, b=1.0):
    """Return n Chebyshev nodes on [a, b]."""
    k = np.arange(1, n+1)
    return 0.5*(b+a) + 0.5*(b-a)*np.cos(np.pi*(2*k-1)/(2*n))


runge_fn = lambda x: 1.0 / (1 + 25*x**2)
x_plot = np.linspace(-1, 1, 400)

n_pts   = 12
x_unif  = np.linspace(-1, 1, n_pts)
x_cheb  = chebyshev_nodes(n_pts)

y_true  = runge_fn(x_plot)
y_unif  = lagrange_interpolate(x_unif, runge_fn(x_unif), x_plot)
y_cheb  = lagrange_interpolate(x_cheb, runge_fn(x_cheb), x_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_interp, nodes, title in zip(
    axes,
    [y_unif, y_cheb],
    [x_unif, x_cheb],
    ['Uniform grid (Runge phenomenon)', 'Chebyshev nodes (well-behaved)']
):
    ax.plot(x_plot, y_true, 'k-', lw=2, label='True f(x)')
    ax.plot(x_plot, y_interp, 'b--', label='Interpolant')
    ax.scatter(nodes, runge_fn(nodes), color='red', zorder=5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

### 1.3 Newton's Divided Differences

The same unique interpolation polynomial can be written in **Newton's form**, which makes adding a new data point cheap (just one extra term):

    pi_n(x) = f[x_1] + f[x_1,x_2](x-x_1) + f[x_1,x_2,x_3](x-x_1)(x-x_2) + ...

The **divided differences** are defined recursively:

    f[x_i, x_{i+1}] = (f(x_{i+1}) - f(x_i)) / (x_{i+1} - x_i)
    f[x_i, ..., x_{i+k}] = (f[x_{i+1},...,x_{i+k}] - f[x_i,...,x_{i+k-1}]) / (x_{i+k} - x_i)

On a uniform grid with step delta, f[x_1, x_2] ≈ f'(x_1), showing that Newton's polynomial is a finite-difference Taylor expansion.

In [ ]:
def newton_divided_differences(x, y):
    """Build divided difference table. Returns Newton coefficient array."""
    n = len(x)
    table = np.zeros((n, n))
    table[:, 0] = y
    for j in range(1, n):
        for i in range(n - j):
            table[i, j] = (table[i+1, j-1] - table[i, j-1]) / (x[i+j] - x[i])
    return table[0, :]  # coefficients c_0, c_1, ..., c_{n-1}


def newton_eval(coeffs, x_nodes, x_eval):
    """Evaluate Newton interpolation polynomial using Horner-like scheme."""
    x_eval = np.asarray(x_eval, float)
    result  = np.full_like(x_eval, coeffs[-1])
    for i in range(len(coeffs)-2, -1, -1):
        result = result * (x_eval - x_nodes[i]) + coeffs[i]
    return result


# Bio application: gene expression interpolation of missing time points
# Simulated qRT-PCR data: 6 measured time points, 2 to be predicted
t_measured = np.array([0, 2, 4, 8, 12, 24], dtype=float)   # hours
expr       = np.array([1.0, 2.3, 4.1, 6.8, 5.2, 3.1])      # log2 fold change

coeffs  = newton_divided_differences(t_measured, expr)
t_dense = np.linspace(0, 24, 200)
expr_interp = newton_eval(coeffs, t_measured, t_dense)

missing_t  = np.array([6.0, 18.0])
expr_pred  = newton_eval(coeffs, t_measured, missing_t)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_dense, expr_interp, label='Newton interpolant')
ax.scatter(t_measured, expr, color='black', zorder=5, label='Measured')
ax.scatter(missing_t, expr_pred, color='red', marker='*', s=150, zorder=6,
           label='Predicted missing points')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('log2 fold change')
ax.set_title('Gene expression: Newton interpolation of missing time points')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Predicted at t=6h: {expr_pred[0]:.2f},  t=18h: {expr_pred[1]:.2f}')

### 1.4 Cubic Spline Interpolation

High-degree polynomial interpolants are unstable. **Splines** avoid this by using low-degree polynomials on each sub-interval, stitched together with continuity conditions.

A **cubic spline** s(x) in S_{3,1}(X, I) is a piecewise cubic with continuous first and second derivatives everywhere. It solves the variational problem:

    min_f  integral_a^b [f''(x)]^2 dx   subject to f(x_i) = y_i

This is the mathematical model of a physical *spline* — a flexible elastic strip bent to pass through fixed points, historically used by shipbuilders.

**Error bound:** For f in W^4[M_4, I]:  ||s - f||_inf <= (5/364) * h^4 * M_4,  where h = max(x_{i+1} - x_i).

In [ ]:
# Bio application: reconstruct a full circadian time course
# Measured at irregular intervals; cubic spline gives a smooth biological curve
t_sampled = np.array([0, 3, 6, 9, 12, 15, 18, 21, 24], dtype=float)  # hours ZT
per1_expr = np.array([1.0, 3.2, 7.1, 9.4, 6.8, 2.9, 1.1, 0.8, 1.0])  # Per1 (AU)

# Periodic boundary condition: same value and derivative at both endpoints
cs = CubicSpline(t_sampled, per1_expr, bc_type='periodic')
t_fine = np.linspace(0, 24, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(t_sampled, per1_expr, color='black', zorder=5, label='Measured')
axes[0].plot(t_fine, cs(t_fine), label='Cubic spline')
axes[0].set_xlabel('ZT (hours)')
axes[0].set_ylabel('Per1 expression (AU)')
axes[0].set_title('Circadian Per1 expression -- cubic spline')
axes[0].legend()

lin = interp1d(t_sampled, per1_expr, kind='linear')
axes[1].plot(t_fine, cs(t_fine), label='Cubic spline')
axes[1].plot(t_fine, lin(t_fine), '--', label='Linear interpolation')
axes[1].scatter(t_sampled, per1_expr, color='black', zorder=5)
axes[1].set_xlabel('ZT (hours)')
axes[1].set_title('Spline vs linear -- smoothness matters')
axes[1].legend()

plt.tight_layout()
plt.show()

peak_t = t_fine[np.argmax(cs(t_fine))]
print(f'Estimated peak expression at ZT {peak_t:.1f}h')

---
## Part 2: Numerical Differentiation and Integration

### 2.1 Finite Difference Formulas

We approximate derivatives by replacing the limiting process with finite differences. Derived from Taylor series:

| Formula | Expression | Error order |
|---------|-----------|-------------|
| Forward difference  | (f(x+h) - f(x)) / h         | O(h)   |
| Backward difference | (f(x) - f(x-h)) / h         | O(h)   |
| Central difference  | (f(x+h) - f(x-h)) / (2h)   | O(h^2) |

**Instability with noisy data:** The total error for a noisy measurement is ~M_2*h + epsilon/h, minimised at h* ~ sqrt(epsilon/M_2). Taking h too small amplifies measurement noise — the classic numerical instability of differentiation.

In [ ]:
def forward_diff(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

def backward_diff(f, x, h=1e-5):
    return (f(x) - f(x - h)) / h

def central_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2*h)


# Error vs step size: central difference is O(h^2), much better
f_test       = np.sin
f_prime_true = np.cos(1.0)
h_vals = np.logspace(-8, 0, 200)

err_fwd  = np.abs([forward_diff(f_test, 1.0, h)  - f_prime_true for h in h_vals])
err_cent = np.abs([central_diff(f_test, 1.0, h)  - f_prime_true for h in h_vals])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_vals, err_fwd,  label='Forward  O(h)')
ax.loglog(h_vals, err_cent, label='Central  O(h^2)')
ax.loglog(h_vals, h_vals,   'k--', alpha=0.5, label='slope 1')
ax.loglog(h_vals, h_vals**2,'k:',  alpha=0.5, label='slope 2')
ax.set_xlabel('Step size h')
ax.set_ylabel('Absolute error')
ax.set_title('Differentiation error vs step size (sin at x=1)')
ax.legend()
plt.tight_layout()
plt.show()

# Bio application: rate of change of drug plasma concentration
print('\n--- Drug clearance rate dC/dt ---')
t_pk = np.array([0, 1, 2, 4, 6, 8, 12, 24], dtype=float)  # hours
C_pk = np.array([100, 72, 52, 27, 14, 7.4, 2.0, 0.1])      # ng/mL
for i in range(1, len(t_pk)-1):
    dCdt = (C_pk[i+1] - C_pk[i-1]) / (t_pk[i+1] - t_pk[i-1])
    print(f'  t={t_pk[i]:4.0f}h: dC/dt = {dCdt:6.2f} ng/mL/h')

### 2.2 Numerical Integration: Trapezoidal Rule and Simpson's Rule

Numerical integration is a **stable** operation (unlike differentiation): measurement errors average out over the interval.

**Trapezoidal rule** (piecewise linear, f in W^2):

    J_N^T = [(f(a)+f(b))/2 + sum_{k=1}^{N-1} f(x_k)] * (b-a)/N
    |J - J_N^T| <= (b-a)^3 / N^2 * M_2

**Simpson's rule** (piecewise quadratic, f in W^4):

    J_N^S = (b-a)/(6N) * [f(a) + f(b) + 4*sum_odd f(x_k) + 2*sum_even f(x_k)]
    |J - J_N^S| <= M_4/(12) * (b-a)^5 / N^4

Simpson's rule converges as O(h^4) vs the trapezoidal O(h^2) — two orders faster.

In [ ]:
def trapezoid_rule(x, y):
    """Composite trapezoidal rule on a non-uniform grid."""
    return np.sum(0.5*(y[1:] + y[:-1]) * np.diff(x))


def simpsons_rule(x, y):
    """Composite Simpson's rule (uniform grid; requires even number of intervals)."""
    n = len(x) - 1
    if n % 2 != 0:
        raise ValueError('Need even number of intervals for Simpson')
    h = (x[-1] - x[0]) / n
    return h/3 * (y[0] + y[-1] + 4*np.sum(y[1:-1:2]) + 2*np.sum(y[2:-2:2]))


# Validation against exact result
x_test = np.linspace(0, np.pi, 101)
y_test = np.sin(x_test)
exact  = 2.0
print(f'True  integral: {exact:.6f}')
print(f'Trapezoid:      {trapezoid_rule(x_test, y_test):.6f}  '
      f'(scipy: {trapezoid(y_test, x_test):.6f})')
print(f'Simpson:        {simpsons_rule(x_test, y_test):.6f}  '
      f'(scipy: {simpson(y_test, x=x_test):.6f})')

# Bio application: area under a dose-response curve (AUDC)
print('\n--- AUC of dose-response curve ---')
dose_log = np.array([-9, -8, -7, -6, -5, -4, -3], dtype=float)  # log10[drug] (M)
response = np.array([2.1, 3.4, 12.8, 48.3, 87.1, 97.2, 99.0])   # % inhibition

auc = trapezoid_rule(dose_log, response)
print(f'AUC (dose-response): {auc:.2f} %·log-units')
print(f'Normalised AUC: {auc / (6.0 * 100.0):.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(dose_log, response, alpha=0.3, label=f'AUC = {auc:.1f}')
ax.plot(dose_log, response, 'o-', color='tab:blue')
ax.set_xlabel('log10[Drug] (M)')
ax.set_ylabel('% Inhibition')
ax.set_title('Dose-response AUC (trapezoidal rule)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve AUC -- a core classification metric in clinical bioinformatics
from sklearn.metrics import roc_curve, auc as sklearn_auc

rng_roc = np.random.default_rng(0)
n_roc   = 200
y_true  = rng_roc.integers(0, 2, n_roc)
y_score = rng_roc.beta(2, 5, n_roc) + y_true * rng_roc.beta(3, 2, n_roc)
y_score = np.clip(y_score, 0, 1)

fpr, tpr, _ = roc_curve(y_true, y_score)
auc_trap    = trapezoid_rule(fpr, tpr)   # our implementation
auc_sklearn = sklearn_auc(fpr, tpr)      # reference

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f'ROC (AUC = {auc_trap:.3f})')
ax.fill_between(fpr, tpr, alpha=0.15)
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curve -- AUC via trapezoidal rule')
ax.legend()
plt.tight_layout()
plt.show()
print(f'AUC (our trapezoid): {auc_trap:.4f} | sklearn: {auc_sklearn:.4f}')

---
## Part 3: Curve Fitting and Least Squares

### 3.1 The Least Squares Problem

Given m measurements b_i = f(t_i) + epsilon_i and a model f_bar(t) = sum_k x_k * phi_k(t) with n << m, the **method of least squares** finds x = (x_1, ..., x_n) minimising:

    Q(x) = sum_i (f_bar(t_i) - b_i)^2 = ||Ax - b||^2

where A_{ik} = phi_k(t_i) is the **design matrix**. The minimum satisfies the **normal equations**:

    A^T A x = A^T b   (unique solution when rank(A) = n)

In practice: use SVD-based solvers (numpy.linalg.lstsq) rather than forming A^T A directly -- the condition number of A^T A equals kappa(A)^2.

In [ ]:
# Linear least squares: polynomial fit to Mendeleev solubility data
# Design matrix for polynomial basis phi_k(t) = t^k
degree = 2
A_design = np.column_stack([T_data**k for k in range(degree+1)])

# numpy.linalg.lstsq uses SVD internally
x_coef, _, rank_A, sv = np.linalg.lstsq(A_design, S_data, rcond=None)
cond_A = sv[0] / sv[-1]
print(f'Polynomial coefficients (a0, a1, a2): {x_coef}')
print(f'Condition number of design matrix: {cond_A:.1f}')

# numpy.polyfit for comparison
p2 = np.polyfit(T_data, S_data, deg=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(T_data, S_data, color='black', zorder=5)
axes[0].plot(T_fine, np.polyval(p2, T_fine), label='Polynomial fit (deg=2)')
axes[0].set_xlabel('Temperature (C)')
axes[0].set_ylabel('Solubility (g/100mL)')
axes[0].set_title('Mendeleev data: least-squares quadratic fit')
axes[0].legend()

S_fitted   = np.polyval(p2, T_data)
residuals  = S_data - S_fitted
axes[1].bar(T_data, residuals, width=2)
axes[1].axhline(0, color='black')
axes[1].set_xlabel('Temperature (C)')
axes[1].set_ylabel('Residual (g/100mL)')
axes[1].set_title('Residuals of quadratic fit')
plt.tight_layout()
plt.show()

### 3.2 Nonlinear Curve Fitting with scipy.optimize.curve_fit

curve_fit minimises sum (y_i - f(x_i, theta))^2 using the Levenberg-Marquardt algorithm (a trust-region variant of Newton's method). The returned pcov matrix is the estimated covariance of the fitted parameters (from the Jacobian at the solution).

Standard errors: sigma_k = sqrt(pcov[k,k]).  95% confidence interval: popt[k] +/- 1.96 * sigma_k.

In [ ]:
def michaelis_menten(S, Vmax, Km):
    """Michaelis-Menten kinetics: v = Vmax*S / (Km + S)"""
    return Vmax * S / (Km + S)


def hill_equation(S, Vmax, K_half, n):
    """Hill equation: v = Vmax*S^n / (K_half^n + S^n)"""
    return Vmax * S**n / (K_half**n + S**n)


def four_pl(x, bottom, top, ec50, hill_n):
    """Four-parameter logistic (4PL): standard for ELISA and dose-response."""
    return bottom + (top - bottom) / (1 + (ec50/x)**hill_n)


# Simulated Michaelis-Menten kinetics data
S_conc = np.array([0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0])
v_true = michaelis_menten(S_conc, Vmax=95.0, Km=0.4)
v_obs  = v_true + rng.normal(0, 2.5, len(S_conc))

popt_mm, pcov_mm = curve_fit(michaelis_menten, S_conc, v_obs,
                             p0=[np.max(v_obs), np.median(S_conc)])
perr_mm  = np.sqrt(np.diag(pcov_mm)) * 1.96
Vmax_fit, Km_fit = popt_mm

# Hill equation fit
popt_h, pcov_h = curve_fit(hill_equation, S_conc, v_obs,
                           p0=[np.max(v_obs), np.median(S_conc), 1.0],
                           maxfev=5000)

S_fine = np.logspace(np.log10(0.01), np.log10(20), 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(S_conc, v_obs, 'o', color='black', label='Data')
axes[0].semilogx(S_fine, michaelis_menten(S_fine, *popt_mm),
    label=f'MM: Vmax={Vmax_fit:.1f}+-{perr_mm[0]:.1f}, Km={Km_fit:.3f}+-{perr_mm[1]:.3f} mM')
axes[0].semilogx(S_fine, hill_equation(S_fine, *popt_h), '--',
    label=f'Hill: n={popt_h[2]:.2f}')
axes[0].set_xlabel('[S] (mM)')
axes[0].set_ylabel('v (nmol/min/mg)')
axes[0].set_title('Enzyme kinetics: MM vs Hill')
axes[0].legend(fontsize=8)

# Hemoglobin O2 binding -- cooperative, Hill n > 1
pO2 = np.array([1, 2, 5, 10, 20, 40, 80, 100], dtype=float)
sat = np.array([5, 10, 28, 52, 75, 90, 97, 98], dtype=float)
popt_hb, _ = curve_fit(hill_equation, pO2, sat, p0=[100.0, 25.0, 2.5])

pO2_fine = np.linspace(0.5, 150, 300)
axes[1].semilogx(pO2, sat, 'o', color='black', label='Hb-O2 data')
axes[1].semilogx(pO2_fine, hill_equation(pO2_fine, *popt_hb),
    label=f'Hill: n={popt_hb[2]:.2f}, P50={popt_hb[1]:.1f} mmHg')
axes[1].set_xlabel('pO2 (mmHg)')
axes[1].set_ylabel('% O2 saturation')
axes[1].set_title('Hemoglobin-O2 binding (cooperative)')
axes[1].legend()
plt.tight_layout()
plt.show()

### 3.3 Goodness-of-Fit Metrics: R², chi-squared, and AIC

After fitting, always ask: does the model describe the data well? Is it the best model?

| Metric | Formula | Notes |
|--------|---------|-------|
| R2    | 1 - SS_res / SS_tot | Fraction of variance explained; 1 = perfect |
| chi2  | sum (y_i - y_hat_i)^2 / sigma_i^2 | Follows chi^2 distribution with n-p dof |
| AIC   | n*ln(SS/n) + 2p | Lower = better; penalises extra parameters |
| AICc  | AIC + 2p(p+1)/(n-p-1) | Small-sample correction |

In [ ]:
def r_squared(y_obs, y_pred):
    ss_res = np.sum((y_obs - y_pred)**2)
    ss_tot = np.sum((y_obs - np.mean(y_obs))**2)
    return 1 - ss_res / ss_tot


def aic(n_params, y_obs, y_pred):
    n   = len(y_obs)
    sse = np.sum((y_obs - y_pred)**2)
    return n * np.log(sse / n) + 2 * n_params


def chi2_gof(y_obs, y_pred, sigma, n_params):
    """Chi-squared goodness-of-fit test. Returns (stat, dof, p_value)."""
    stat  = np.sum(((y_obs - y_pred) / sigma)**2)
    dof   = len(y_obs) - n_params
    pval  = 1 - chi2.cdf(stat, dof)
    return stat, dof, pval


# Compare Michaelis-Menten vs Hill on the kinetics data
v_mm_pred   = michaelis_menten(S_conc, *popt_mm)
v_hill_pred = hill_equation(S_conc, *popt_h)
sigma_est   = np.full_like(v_obs, 2.5)

models = {
    'Michaelis-Menten (2 params)': (v_mm_pred,   2),
    'Hill equation  (3 params)':   (v_hill_pred,  3),
}
print(f'{"Model":<32} {"R2":>6} {"AIC":>8} {"chi2/dof":>9} {"p(chi2)":>9}')
print('-' * 68)
for name, (pred, n_p) in models.items():
    r2 = r_squared(v_obs, pred)
    a  = aic(n_p, v_obs, pred)
    chi2_stat, dof, pval = chi2_gof(v_obs, pred, sigma_est, n_p)
    print(f'{name:<32} {r2:>6.4f} {a:>8.2f} {chi2_stat/dof:>9.2f} {pval:>9.4f}')

print('\nInterpretation: DELTA_AIC > 2 favours the lower-AIC model; DELTA_AIC > 10 is decisive.')

---
## Part 4: Optimization

### 4.1 Gradient Descent from Scratch

The **gradient** nabla Q(x) points in the direction of steepest ascent of Q. Moving in the opposite direction decreases Q:

    x^(k+1) = x^(k) - alpha * nabla Q(x^(k))

where alpha is the **learning rate** (step size). At convergence, nabla Q = 0.

**Challenges:** local minima, saddle points, slow convergence on ravine-shaped loss surfaces. Gelfand's *ravine method* addresses the last issue by tracking two gradient-descent paths simultaneously and searching along the direction connecting them.

In [ ]:
def gradient_descent(grad_Q, x0, lr=0.01, tol=1e-6, max_iter=5000):
    """
    Minimise Q(x) by gradient descent.
    grad_Q  : callable, returns gradient at x
    x0      : initial parameter vector
    Returns : (x_opt, gradient_norm_history)
    """
    x       = np.asarray(x0, dtype=float)
    history = []
    for _ in range(max_iter):
        g = grad_Q(x)
        x = x - lr * g
        history.append(np.linalg.norm(g))
        if history[-1] < tol:
            break
    return x, np.array(history)


# Example: linear regression via gradient descent
x_lin = np.linspace(0, 10, 50)
y_lin = 2.3 * x_lin + 1.5 + rng.normal(0, 1.5, 50)

def mse_grad(params):
    a, b = params
    y_hat = a * x_lin + b
    n     = len(x_lin)
    da    = (2/n) * np.sum((y_hat - y_lin) * x_lin)
    db    = (2/n) * np.sum(y_hat - y_lin)
    return np.array([da, db])

x_opt, conv = gradient_descent(mse_grad, x0=[0.0, 0.0], lr=0.005, tol=1e-8)
a_opt, b_opt = x_opt

A_lin  = np.column_stack([x_lin, np.ones_like(x_lin)])
x_lstsq, *_ = np.linalg.lstsq(A_lin, y_lin, rcond=None)

print(f'GD result:    a={a_opt:.4f}, b={b_opt:.4f}')
print(f'lstsq result: a={x_lstsq[0]:.4f}, b={x_lstsq[1]:.4f}')
print(f'Converged in {len(conv)} iterations')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x_lin, y_lin, s=15, alpha=0.7)
axes[0].plot(x_lin, a_opt*x_lin + b_opt, 'r-', label='GD fit')
axes[0].set_title('Linear regression via gradient descent')
axes[0].legend()

axes[1].semilogy(conv)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('||gradient||')
axes[1].set_title('Gradient descent convergence')
plt.tight_layout()
plt.show()

### 4.2 scipy.optimize.minimize and Maximum Likelihood Estimation

scipy.optimize.minimize provides a unified interface to many algorithms:

| Method | Requires gradient | Best for |
|--------|-----------------|----------|
| Nelder-Mead  | No              | Small dimension, noisy functions |
| BFGS         | Optional        | Smooth, unconstrained |
| L-BFGS-B     | Optional        | Large-scale, box constraints |
| SLSQP        | Yes             | Constrained problems |
| trust-constr | Yes             | General constrained |

Always check result.success and inspect result.message after calling minimize.

In [ ]:
# MLE for a normal distribution -- maximise log-likelihood over mu and sigma
gene_expr = rng.normal(loc=5.2, scale=1.8, size=80)  # simulated log2 expression values

def neg_log_likelihood_normal(params):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)  # log-parameterisation keeps sigma strictly positive
    return (0.5 * len(gene_expr) * np.log(2*np.pi*sigma**2)
            + np.sum((gene_expr - mu)**2) / (2*sigma**2))


result_nm = minimize(neg_log_likelihood_normal, x0=[4.0, 0.5],
                     method='Nelder-Mead',
                     options={'xatol': 1e-8, 'fatol': 1e-8})
mu_mle    = result_nm.x[0]
sigma_mle = np.exp(result_nm.x[1])

print(f'True parameters:    mu=5.20, sigma=1.80')
print(f'MLE (Nelder-Mead):  mu={mu_mle:.3f}, sigma={sigma_mle:.3f}')
print(f'Sample mean: {gene_expr.mean():.3f}, sample std (ddof=1): {gene_expr.std(ddof=1):.3f}')

# Visualise the negative log-likelihood surface
mu_grid    = np.linspace(4.0, 6.5, 80)
sigma_grid = np.linspace(0.8, 3.0, 80)
MU, SIG = np.meshgrid(mu_grid, sigma_grid)
NLL = np.array([[neg_log_likelihood_normal([m, np.log(s)])
                 for m in mu_grid] for s in sigma_grid])

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(MU, SIG, NLL, levels=30, cmap='viridis')
plt.colorbar(cf, ax=ax, label='Negative log-likelihood')
ax.scatter(mu_mle, sigma_mle, color='red', s=80, zorder=5, label='MLE')
ax.set_xlabel('mu')
ax.set_ylabel('sigma')
ax.set_title('NLL surface -- Nelder-Mead finds the minimum')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# MLE for negative binomial distribution -- models overdispersed RNA-seq read counts
from scipy.stats import nbinom

true_r, true_p = 5.0, 0.4
counts = nbinom.rvs(true_r, true_p, size=200, random_state=1)

def neg_ll_nb(params):
    log_r, logit_p = params
    r = np.exp(log_r)
    p = 1.0 / (1.0 + np.exp(-logit_p))
    return -np.sum(nbinom.logpmf(counts, r, p))


result_nb = minimize(neg_ll_nb, x0=[np.log(3.0), 0.0], method='Nelder-Mead',
                     options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 5000})
r_fit = np.exp(result_nb.x[0])
p_fit = 1.0 / (1.0 + np.exp(-result_nb.x[1]))

print(f'True:  r={true_r:.2f}, p={true_p:.3f}')
print(f'MLE:   r={r_fit:.2f}, p={p_fit:.3f}')
print(f'Observed: mean={counts.mean():.1f}, var={counts.var():.1f}')
nb_mean = r_fit*(1-p_fit)/p_fit
nb_var  = r_fit*(1-p_fit)/p_fit**2
print(f'NB fitted: mean={nb_mean:.1f}, var={nb_var:.1f}')

x_cnt = np.arange(0, counts.max()+1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_cnt, np.bincount(counts, minlength=len(x_cnt))/len(counts),
       alpha=0.6, label='Observed counts')
ax.plot(x_cnt, nbinom.pmf(x_cnt, r_fit, p_fit), 'r-o', ms=4, label='Fitted NB')
ax.set_xlabel('Read count')
ax.set_ylabel('Frequency')
ax.set_title('RNA-seq counts: MLE for negative binomial')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 5: Fourier Transform

### 5.1 Fourier Series and the Continuous Fourier Transform

Any periodic function f(t) with period 2*pi decomposes into a **Fourier series**:

    f(t) = a_0/2 + sum_{n=1}^inf [a_n*cos(n*t) + b_n*sin(n*t)]

For non-periodic functions, the **Fourier transform** generalises this to a continuous spectrum:

    f_hat(omega) = integral_{-inf}^{+inf} e^{-i*omega*t} f(t) dt          (forward)
    f(t) = 1/(2*pi) * integral_{-inf}^{+inf} e^{i*omega*t} f_hat(omega) domega  (inverse)

Key properties:
- Differentiation: FT{f'}(omega) = -i*omega * f_hat(omega) -- differentiation becomes multiplication in frequency space
- Convolution: FT{f1 * f2} = f1_hat * f2_hat -- convolution becomes pointwise multiplication
- Uncertainty principle: L(f) * L(f_hat) ~ 1 -- narrow in time implies broad in frequency

### 5.2 Discrete Fourier Transform (DFT) and the FFT

With N discrete samples f_n = f(n*dt), the DFT is:

    A_j = (1/N) * sum_{n=0}^{N-1} exp(-i*2*pi*n*j/N) * a_n,   j = 0, ..., N-1

**Nyquist criterion:** Maximum detectable frequency is f_max = 1/(2*dt). Frequencies above this cause **aliasing**.

**FFT (Cooley-Tukey, 1965):** O(N log N) instead of O(N^2). This made DFT practical at large N and is implemented in numpy.fft.

In [ ]:
# Bio application: detect circadian period in a gene expression time series
# Simulated microarray time course with 24h and 12h components plus noise
dt   = 2.0          # hours between measurements
N_ts = 72           # total of 144 hours
t_ts = np.arange(N_ts) * dt

# Ground truth: 24h circadian + 12h ultradian + noise
signal = (2.5 * np.sin(2*np.pi*t_ts/24 + 0.3)
        + 1.0 * np.sin(2*np.pi*t_ts/12)
        + rng.normal(0, 0.5, N_ts))

# FFT via numpy
fft_vals = np.fft.rfft(signal)
freqs    = np.fft.rfftfreq(N_ts, d=dt)     # cycles per hour
periods  = np.where(freqs > 0, 1.0/freqs, np.inf)
power    = np.abs(fft_vals)**2

top3_idx = np.argsort(power[1:])[-3:] + 1  # skip DC
print('Top spectral peaks:')
for idx in sorted(top3_idx, reverse=True):
    print(f'  Period = {periods[idx]:.1f} h  |  Power = {power[idx]:.1f}')

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(t_ts, signal)
axes[0].set_xlabel('Time (hours)')
axes[0].set_ylabel('Expression')
axes[0].set_title('Simulated circadian gene expression (Per1-like)')

axes[1].plot(periods[1:N_ts//2], power[1:N_ts//2])
for idx in top3_idx:
    if periods[idx] < 100:
        axes[1].axvline(periods[idx], color='red', linestyle='--', alpha=0.7,
                       label=f'{periods[idx]:.0f}h')
axes[1].set_xlim(0, 50)
axes[1].set_xlabel('Period (hours)')
axes[1].set_ylabel('Power')
axes[1].set_title('Power spectrum -- peaks at 24h and 12h')
axes[1].legend()
plt.tight_layout()
plt.show()

### 5.3 Fourier Filtering and the 2D FFT

In Fourier space, **low-pass filtering** removes high-frequency components (noise). Algorithm:

    f(t) -> FFT -> f_hat(omega) -> multiply by R(omega) -> IFFT -> filtered f(t)

This is exactly the mathematical smoothing operation described in Mendeleev's numerical methods course.

The **2D Fourier transform** is used in:
- X-ray crystallography: structure factors F(hkl) are Fourier coefficients of the electron density rho(x,y,z)
- Cryo-EM: Fourier-based alignment and reconstruction
- MRI: k-space data is the 2D Fourier transform of the image

In [ ]:
# Fourier filtering: denoise a simulated NMR free induction decay (FID)
fs_nmr  = 1000.0   # Hz sampling frequency
N_nmr   = 2048
t_nmr   = np.arange(N_nmr) / fs_nmr

# Two protein resonances + broadband noise
fid_signal = (3.0 * np.sin(2*np.pi*120*t_nmr) * np.exp(-t_nmr*8)
            + 1.5 * np.sin(2*np.pi*340*t_nmr) * np.exp(-t_nmr*12)
            + rng.normal(0, 0.8, N_nmr))

fft_fid  = np.fft.rfft(fid_signal)
freqs_nm = np.fft.rfftfreq(N_nmr, d=1/fs_nmr)
power_nm = np.abs(fft_fid)**2

# Low-pass filter: zero out frequencies above 450 Hz
cutoff_hz = 450
fft_filt  = fft_fid.copy()
fft_filt[freqs_nm > cutoff_hz] = 0
fid_filtered = np.fft.irfft(fft_filt, n=N_nmr)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].plot(t_nmr[:200], fid_signal[:200])
axes[0,0].set_title('Raw FID signal (time domain)')
axes[0,0].set_xlabel('Time (s)')

axes[0,1].semilogy(freqs_nm[1:], power_nm[1:])
axes[0,1].axvline(120, color='r', alpha=0.6, label='120 Hz')
axes[0,1].axvline(340, color='g', alpha=0.6, label='340 Hz')
axes[0,1].set_xlim(0, 500)
axes[0,1].set_xlabel('Frequency (Hz)')
axes[0,1].set_ylabel('Power')
axes[0,1].set_title('NMR power spectrum')
axes[0,1].legend()

axes[1,0].plot(t_nmr[:200], fid_signal[:200], alpha=0.4, label='Noisy')
axes[1,0].plot(t_nmr[:200], fid_filtered[:200], lw=2, label='Low-pass filtered')
axes[1,0].set_xlabel('Time (s)')
axes[1,0].set_title('Noise removal via Fourier low-pass filter')
axes[1,0].legend()

# 2D FFT: simulate electron density map (two Gaussian atoms)
x2d = np.linspace(-10, 10, 128)
X2, Y2 = np.meshgrid(x2d, x2d)
rho = (np.exp(-((X2-2)**2 + (Y2-1)**2)/2)
     + 0.6*np.exp(-((X2+2)**2 + (Y2+2)**2)/1.5))
F_hkl = np.fft.fftshift(np.fft.fft2(rho))

axes[1,1].imshow(np.log1p(np.abs(F_hkl)**2), cmap='inferno')
axes[1,1].set_title('2D FFT: diffraction pattern |F(hkl)|^2 (log)')
axes[1,1].set_xlabel('h')
axes[1,1].set_ylabel('k')

plt.tight_layout()
plt.show()

# Reconstruction from structure factors
rho_back = np.real(np.fft.ifft2(np.fft.ifftshift(F_hkl)))
print(f'Max reconstruction error (2D FFT roundtrip): {np.max(np.abs(rho - rho_back)):.2e}')
print(f'Peak NMR frequencies: {freqs_nm[np.argsort(power_nm)[-3:-1]][::-1]} Hz')

---
## Exercises

### Exercise 1: Protein Melting Curve -- Spline vs Polynomial

DSF (differential scanning fluorimetry) data: fluorescence signal during thermal unfolding.
`T = [25, 35, 42, 48, 52, 56, 60, 70]` C, `F = [120, 118, 125, 180, 420, 680, 710, 715]` AU.

1. Fit a cubic spline and a degree-7 Lagrange polynomial to the data
2. Plot both on the range 25-70 C
3. Estimate Tm (inflection point) from the spline using `cs.derivative()(T_fine)` and finding its maximum
4. Is the Lagrange polynomial well-behaved? Explain why or why not.

In [ ]:
T_dsf = np.array([25, 35, 42, 48, 52, 56, 60, 70], dtype=float)
F_dsf = np.array([120, 118, 125, 180, 420, 680, 710, 715], dtype=float)

# Your code here
# cs_dsf = CubicSpline(T_dsf, F_dsf)
# dF_dT  = cs_dsf.derivative()(np.linspace(25, 70, 400))
# Tm     = np.linspace(25, 70, 400)[np.argmax(dF_dT)]

### Exercise 2: Pharmacokinetic AUC

Plasma concentration after IV bolus: `t_hr = [0, 0.5, 1, 2, 4, 6, 8, 12, 24]` h,
`C_ng = [500, 380, 290, 170, 58, 20, 7.0, 0.9, 0.01]` ng/mL.

1. Compute AUC_0-24 using the trapezoidal rule
2. Fit a two-compartment model C(t) = A*exp(-alpha*t) + B*exp(-beta*t) with curve_fit
3. Compute AUC_0-inf analytically: A/alpha + B/beta
4. Compare the two AUC values

In [ ]:
t_pk2 = np.array([0, 0.5, 1, 2, 4, 6, 8, 12, 24], dtype=float)
C_ng2 = np.array([500, 380, 290, 170, 58, 20, 7.0, 0.9, 0.01])

# Your code here
# def two_compartment(t, A, alpha, B, beta): return A*np.exp(-alpha*t) + B*np.exp(-beta*t)

### Exercise 3: Cell Viability Dose-Response and AIC Model Selection

`conc_uM = [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]` uM
`viability = [98, 95, 87, 68, 42, 18, 6, 2]` %

1. Fit 3-parameter Hill: v = 100 / (1 + (IC50/x)^n)
2. Fit 4PL: v = bottom + (top - bottom) / (1 + (EC50/x)^n)
3. Compare R2 and AIC; report IC50 and Hill coefficient with 95% CI

In [ ]:
conc_uM   = np.array([0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0])
viability = np.array([98, 95, 87, 68, 42, 18, 6, 2], dtype=float)

# Your code here

### Exercise 4: Logistic Regression via Gradient Descent

Binary classification of gene expression profiles (benign vs malignant).

1. Generate: 20 benign samples ~ N([1,1], 0.8), 20 malignant ~ N([3,3], 0.8)
2. Implement cross-entropy loss and its gradient; run gradient descent with lr=0.1 for 1000 iterations
3. Plot loss curve and decision boundary
4. Compare with scipy.optimize.minimize(method='BFGS')

In [ ]:
rng_ex = np.random.default_rng(5)
X0 = rng_ex.normal([1,1], 0.8, (20,2))
X1 = rng_ex.normal([3,3], 0.8, (20,2))
X_cls = np.vstack([X0, X1])
y_cls = np.array([0]*20 + [1]*20, dtype=float)

# Your code here
# sigmoid = lambda z: 1.0 / (1.0 + np.exp(-z))
# cross_entropy gradient: X.T @ (sigmoid(X @ w) - y) / n

### Exercise 5: Periodicity Detection in Yeast Cell Cycle Data

25 time points, sampled every 5 minutes (125 min total). One gene is cell-cycle regulated, two are constitutive.

1. Compute power spectra for all three genes
2. Find which gene has a peak near the expected yeast cell cycle period (~70 min)
3. Compute a periodicity score: max_power / mean_power
4. Apply band-pass filter around 60-80 min and plot the filtered signal

In [ ]:
rng_cc = np.random.default_rng(99)
t_cc = np.arange(25) * 5.0    # minutes
T_cc = 70.0                    # yeast cell cycle ~70 min

gene_periodic = 2.0*np.sin(2*np.pi*t_cc/T_cc) + 0.5*rng_cc.normal(0, 1, 25)
gene_const1   = 1.0 + rng_cc.normal(0, 0.4, 25)
gene_const2   = 0.5*np.sin(2*np.pi*t_cc/10) + rng_cc.normal(0, 0.3, 25)  # high freq

# Your code here

### Exercise 6: Complete Michaelis-Menten Workflow with Replicates

```
S_rep = [0.05, 0.05, 0.1, 0.1, 0.2, 0.2, 0.5, 0.5, 1.0, 1.0, 5.0, 5.0]  mM
v_rep = [8.1, 9.3, 15.2, 14.8, 25.1, 26.9, 48.2, 46.0, 61.3, 63.1, 78.9, 80.1]  nmol/min/mg
```

1. Compute mean and SEM for each substrate concentration
2. Fit Michaelis-Menten; report Vmax and Km with 95% CI from pcov
3. Plot data with error bars and 95% CI band on the fitted curve
4. Plot residuals and check for systematic patterns

In [ ]:
S_rep = np.array([0.05,0.05,0.1,0.1,0.2,0.2,0.5,0.5,1.0,1.0,5.0,5.0])
v_rep = np.array([8.1,9.3,15.2,14.8,25.1,26.9,48.2,46.0,61.3,63.1,78.9,80.1])

# Your code here
# from scipy.stats import sem
# S_unique = np.unique(S_rep)

---
## Summary

| Topic | Key tool | Bio application |
|-------|----------|-----------------|
| Lagrange interpolation | from scratch | Solubility data, missing time points |
| Chebyshev nodes | from scratch | Avoids Runge oscillations |
| Newton divided differences | from scratch | Incremental interpolation |
| Cubic splines | scipy.interpolate.CubicSpline | Circadian expression curves |
| Forward / central differences | from scratch | Drug clearance rate |
| Trapezoidal rule | scipy.integrate.trapezoid | AUC, ROC, pharmacokinetics |
| Simpson's rule | scipy.integrate.simpson | Higher-accuracy integration |
| Linear least squares | numpy.linalg.lstsq | Polynomial regression |
| Nonlinear fitting | scipy.optimize.curve_fit | MM, Hill, 4PL models |
| Goodness-of-fit | R2, AIC, chi2 | Model comparison |
| Gradient descent | from scratch | Linear and logistic regression |
| General optimisation | scipy.optimize.minimize | MLE, NB distribution |
| FFT | numpy.fft | Circadian period detection |
| Fourier filtering | numpy.fft | NMR noise removal |
| 2D FFT | numpy.fft.fft2 | Crystallography, cryo-EM |

**Key principles from the source course (Mendeleev Institute, MSU):**
- Interpolation and least squares are fundamentally different: interpolation passes through every point; least squares minimises total squared deviation to handle noise
- Numerical differentiation is unstable (error grows as epsilon/h); numerical integration is stable (errors cancel)
- High condition number of A^T A signals an ill-conditioned fit; SVD-based lstsq is more reliable than forming normal equations directly
- The error of interpolation by splines does not grow with the number of knots, unlike high-degree polynomials
- The FFT enabled practical DFT for large N; the Cooley-Tukey algorithm (O(N log N)) is the foundation of modern spectral analysis in NMR and crystallography